# 🛡️ BƯỚC 1: HERETIC UNCENSORING PIPELINE (Chạy trên Colab GPU A100 / L4 / T4)
Notebook này thực hiện **loại bỏ 100% kiểm duyệt / từ chối trả lời (Censorship / Refusal)** cho các mô hình Coding bằng công cụ **Heretic**.

### 📌 Quy trình hoàn toàn tự động (Non-Interactive):
1. Tự động kiểm tra GPU, gỡ bỏ `torchaudio` lệch CUDA và cài đặt `heretic-llm`.
2. Kết nối Google Drive lưu trữ model.
3. Tự động sinh file `config.toml` cấu hình Heretic chạy tự động và lưu thẳng vào Google Drive.
4. Kiểm tra file model đã hoàn tất trên Google Drive (`/content/drive/MyDrive/ai_coding_models_uncensored/`).
5. (Tùy chọn) Nhập Token Hugging Face để tự động đẩy model lên tài khoản của bạn.

In [ ]:
# @title 1. Tự động Tối ưu Môi trường & Cài đặt Heretic LLM
import torch
!nvidia-smi

# Gỡ bỏ torchaudio bị lệch CUDA phiên bản để tránh lỗi BloomPreTrainedModel
!pip uninstall -y torchaudio

# Cài đặt Heretic và các thư viện cần thiết
!pip install -q -U heretic-llm torch torchvision transformers accelerate bitsandbytes huggingface_hub optuna

In [ ]:
# @title 2. Kết nối Google Drive để lưu trữ Model
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# Tạo thư mục chứa các model uncensored trên Drive
SAVE_DIR = "/content/drive/MyDrive/ai_coding_models_uncensored"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"📁 Thư mục lưu trữ model đã uncensor: {SAVE_DIR}")

In [ ]:
# @title 3. Chạy Heretic Uncensoring Tự Động (Non-Interactive qua config.toml)
# @markdown Chọn model lập trình bạn muốn xử lý:
MODEL_CHOICE = "Qwen/Qwen2.5-Coder-7B-Instruct" # @param ["Qwen/Qwen2.5-Coder-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"]
USE_4BIT_QUANT = False # @param {type:"boolean"} # Khuyên dùng: False khi chạy trên A100 40GB/80GB để giữ độ chính xác cao nhất
N_TRIALS = 10 # @param {type:"integer"}

clean_name = MODEL_CHOICE.split("/")[-1]
OUTPUT_MODEL_NAME = f"{clean_name}-Heretic-Uncensored"
TARGET_PATH = os.path.join(SAVE_DIR, OUTPUT_MODEL_NAME)

print(f"🎯 Model mục tiêu: {MODEL_CHOICE}")
print(f"📁 Thư mục lưu trực tiếp trên Drive: {TARGET_PATH}")

# Tạo file config.toml tự động hóa 100% để Heretic không hỏi phím bấm
quant_val = "bnb_4bit" if USE_4BIT_QUANT else "none"
config_content = f'''
model = "{MODEL_CHOICE}"
quantization = "{quant_val}"
model_action = "save"
save_directory = "{TARGET_PATH}"
trial_index = 0
export_strategy = "merge"
n_trials = {N_TRIALS}
'''

with open("config.toml", "w") as f:
    f.write(config_content.strip())

print("⚙️ File config.toml đã được thiết lập tự động:")
print(config_content.strip())
print("\n🚀 Bắt đầu thực thi Heretic Uncensor...")
!heretic

In [ ]:
# @title 4. Kiểm tra Model đã được lưu trên Google Drive
import os

if os.path.exists(TARGET_PATH) and os.path.exists(os.path.join(TARGET_PATH, "config.json")):
    print("🎉 CHÚC MỪNG! Model đã được Uncensor và lưu thành công 100% trên Google Drive!")
    print(f"👉 Đường dẫn lưu trữ: {TARGET_PATH}")
    print("\nDanh sách các file đã tạo:")
    for f in sorted(os.listdir(TARGET_PATH)):
        size_mb = os.path.getsize(os.path.join(TARGET_PATH, f)) / (1024 * 1024)
        print(f"  - {f} ({size_mb:.1f} MB)")
else:
    print(f"⚠️ Đang kiểm tra thư mục: {TARGET_PATH}")
    if os.path.exists(TARGET_PATH):
        print("📁 Files hiện có:", os.listdir(TARGET_PATH))
    else:
        print("❌ Chưa tìm thấy thư mục model. Vui lòng kiểm tra lại log chạy ở Cell 3.")

In [ ]:
# @title 5. (Tùy chọn) Đẩy Model lên Hugging Face Hub
# @markdown Dán Access Token Hugging Face của bạn (bắt đầu bằng hf_...):
HF_TOKEN = "" # @param {type:"string"}

if HF_TOKEN.strip() and os.path.exists(TARGET_PATH):
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN.strip())
    try:
        username = api.whoami()["name"]
        print(f"👤 Tài khoản Hugging Face: {username}")
    except Exception:
        username = "Leon234aamon"
        print(f"👤 Sử dụng tài khoản: {username}")

    HF_REPO_ID = f"{username}/{OUTPUT_MODEL_NAME}"

    print(f"📤 Đang tạo repo private và tải model lên: {HF_REPO_ID}...")
    api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)
    api.upload_folder(
        folder_path=TARGET_PATH,
        repo_id=HF_REPO_ID,
        repo_type="model"
    )
    print(f"🎉 Upload hoàn tất thành công! Xem model tại: https://huggingface.co/{HF_REPO_ID}")
else:
    print("ℹ️ Bỏ qua upload Hugging Face (Model đã được lưu an toàn trên Google Drive của bạn).")